In [16]:
import sys
sys.path.append(r"C:\Users\Rakesh\Documents\summer-project")
sys.path.append(r"C:\Users\Rakesh\Documents\summer-project\src")

from functions_v2 import *
from two_level_mc import TwoLevelSetup
from graph_mlmc_model import GraphTwoLevelModel
from mlmc_runner import MLMCRunner
from sksparse.cholmod import cholesky as sparse_cholesky

In [19]:
edges, n_vertices, edge_weights = load_graph(r"../../data/raw/power-US-Grid.mtx")
num_components = check_connectivity(edges, n_vertices)
if num_components > 1:
    edges, n_vertices, edge_weights = filter_to_largest_component(edges, n_vertices, edge_weights)

A, D, L = build_graph_matrices(edges, n_vertices)
lambda_min = compute_lambda_min(L, D)
L_sigma = build_shifted_laplacian(L, D, lambda_min)

import networkx as nx
G_nx = nx.Graph()
G_nx.add_edges_from(edges)
gamma_in, gamma_out, diameter = find_diameter_endpoints(G_nx, n_sample=5, k_hop=4)

✓ Loaded: ../../data/raw/power-US-Grid.mtx
  Vertices : 4941
  Edges    : 6594
  Weighted : no

Components: 1
  -> Graph is fully connected, safe to proceed

✓ Phase 1 complete: A, D, L built as sparse matrices
  Matrix size : 4941 x 4941
  Degree range: [1, 19]
  Non-zeros in L: 18129

✓ lambda_min = 0.000271
  (eigenvalues found: [0.         0.00027102])
✓ Phase 2 complete: L_sigma built (sigma^2 = 0.25)
  L_sigma type: sparse



In [20]:
setup = TwoLevelSetup.build(
    edges, n_vertices, L_sigma, lambda_min,
    gamma_in, gamma_out,
    build_incidence_matrix, sparse_cholesky,
    max_size=10
)

gamma_in: 23, gamma_out: 14, n_vertices: 4941
Boundary fraction: 0.7488%
  -> Likely safe for aggregation (comparable to validated successes).
n_coarse: 2074 (2074 aggregates)
Size distribution -- min: 2, max: 10, mean: 2.38
Singletons: 0 (0.0%)
gamma_in_coarse: 15, gamma_out_coarse: 7
Overlap (must be empty): set()
Interior coarse vertices: 2052 (98.9%)
coarse edges: 3282 (from 6594 fine edges)


In [23]:
model = GraphTwoLevelModel(setup)
runner = MLMCRunner(model, base_seed=0)
result = runner.run_fixed(samples_per_level=[2000, 300])

print(f"Estimate: {result.estimate:.6f}")
print(f"Standard error: {result.standard_error:.6f}")

Estimate: 0.068223
Standard error: 0.000058


In [27]:
outlet_edges_fine = [(i,j) for i,j in setup.edges if i in setup.gamma_out_set or j in setup.gamma_out_set]
print(f"Outlet edges: {len(outlet_edges_fine)}")
print(f"Their raw estimate: {result.estimate:.6f}")
print(f"Their estimate normalized: {result.estimate / len(outlet_edges_fine):.6f}")

Outlet edges: 23
Their raw estimate: 0.068223
Their estimate normalized: 0.002966


In [29]:
print(f"gamma_in: {len(gamma_in)}, gamma_out: {len(gamma_out)}")
print(f"setup.gamma_in: {len(setup.gamma_in)}, setup.gamma_out: {len(setup.gamma_out)}")

gamma_in: 23, gamma_out: 14
setup.gamma_in: 23, setup.gamma_out: 14


In [33]:
from two_level_mc import *

In [35]:
qf, qc = run_one_paired_sample(setup, seed=0)
print(f"Direct Q_fine: {qf:.6f}, Q_coarse: {qc:.6f}")

Direct Q_fine: 0.152887, Q_coarse: 0.237268


In [37]:
import numpy as np

model = GraphTwoLevelModel(setup)
rng = np.random.default_rng(0)

randomness = model.sample_randomness(1, rng)
coupled = model.couple_inputs(1, randomness)

fine_system = model.build_linear_system(1, coupled.fine)
print("fine_system.A shape:", fine_system.A.shape)
print("fine_system.b shape:", fine_system.b.shape)

from linear_solver import solve_linear_system
fine_result = solve_linear_system(fine_system)
print("fine solve success:", fine_result.success)

Q_fine_adapter = model.quantity_of_interest(1, fine_result.solution, coupled.fine)
print("Q_fine via adapter:", Q_fine_adapter)

fine_system.A shape: (4904, 4904)
fine_system.b shape: (4904,)
fine solve success: True
Q_fine via adapter: 0.15351712727303946


In [39]:
import inspect
print(inspect.signature(MLMCRunner.run_fixed))
print(inspect.getsource(MLMCRunner.run_fixed))

(self, samples_per_level: collections.abc.Sequence[int], *, finest_level: int | None = None) -> mlmc_runner.MLMCResult
    def run_fixed(
        self,
        samples_per_level: Sequence[int],
        *,
        finest_level: int | None = None,
    ) -> MLMCResult:
        """Start a fixed run through the selected or model-finest level."""
        if self._finest_level is not None:
            raise RuntimeError(
                "A run is already active; use add_samples() or create "
                "a new runner."
            )

        resolved_finest_level = self._resolve_finest_level(finest_level)
        counts = self._validate_initial_sample_counts(
            samples_per_level,
            resolved_finest_level,
        )
        self._initialize_run(resolved_finest_level)

        for statistics, sample_count in zip(
            self._level_statistics,
            counts,
            strict=True,
        ):
            self._run_sample_batch(
                statistics,
     

In [41]:
print(model.number_of_levels)

2


In [43]:
print(inspect.getsource(MLMCRunner._resolve_finest_level))
print("---")
print(inspect.getsource(MLMCRunner._run_sample_batch))
print("---")
print(inspect.getsource(MLMCRunner._create_result))

    def _resolve_finest_level(
        self,
        finest_level: int | None,
    ) -> int:
        """Resolve ``None`` to the model's finest level and validate it."""
        if finest_level is None:
            return int(self._model.number_of_levels) - 1

        if isinstance(finest_level, bool) or not isinstance(
            finest_level, Integral
        ):
            raise TypeError("finest_level must be an integer or None.")

        finest_level = int(finest_level)
        if (
            finest_level < 0
            or finest_level >= self._model.number_of_levels
        ):
            raise ValueError(
                "finest_level must be between 0 and "
                f"{self._model.number_of_levels - 1}."
            )

        return finest_level

---
    def _run_sample_batch(
        self,
        level_statistics: LevelStatistics,
        *,
        start_index: int,
        sample_count: int,
    ) -> None:
        """Add a consecutive batch of samples to one cor

In [45]:
import inspect
from mlmc_runner import MLMCResult
print(inspect.getsource(MLMCResult))

@dataclass(frozen=True)
class MLMCResult:
    """Immutable correction-level snapshots and MLMC estimates."""

    finest_level: int
    level_results: tuple[LevelResult, ...]

    @property
    def estimate(self) -> float:
        """Return the sum of the correction-level sample means."""
        return sum(
            (
                result.mean_correction
                for result in self.level_results
            ),
            start=0.0,
        )

    @property
    def estimator_variance(self) -> float:
        """Return the estimated sampling variance of the MLMC mean."""
        return sum(
            (
                result.variance_of_mean
                for result in self.level_results
            ),
            start=0.0,
        )

    @property
    def standard_error(self) -> float:
        """Return the estimated standard error of the MLMC mean."""
        return sqrt(self.estimator_variance)

    @property
    def total_cost(self) -> float:
        """Return the t

In [47]:
for lr in result.level_results:
    print(f"level={lr.level}, n={lr.sample_count}, mean_correction={lr.mean_correction:.6f}, "
          f"mean_cost={lr.mean_sample_cost:.6f}")

level=0, n=2000, mean_correction=0.152103, mean_cost=0.034993
level=1, n=300, mean_correction=-0.083880, mean_cost=0.061461
